# LSM Cloud Training & Model Export (Google Colab)

This notebook allows you to train the heavy LSM (Mexican Sign Language) translation models using Google Colab's GPU acceleration and export them to ONNX format. This ensures compatibility and easy integration with the desktop GUI application.

### Instructions:
1. **Enable GPU**: Go to **Runtime > Change runtime type** and select **T4 GPU** (or any available GPU).
2. Prepare your training dataset (containing `data/static_letters`, `data/dynamic_letters`, and `data/words`) as a ZIP file (e.g. `LSM_Dataset.zip`) and upload it to your Google Drive.
3. Run the cells sequentially.

### 1. Mount Google Drive
Mount your Google Drive to load the dataset and save the trained models.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Clone Git Repository
Clone the LSM repository to get access to the training scripts (`train_models.py`, `utils.py`, etc.).

In [ ]:
!git clone https://github.com/lgbmA01731203/LSM.git
%cd LSM

### 3. Install Requirements
Install the required packages for Colab training.

In [ ]:
!pip install -r requirements_colab.txt

### 4. Extract Dataset from Google Drive
Unzip your dataset file from Google Drive into the workspace. Modify the file path if your dataset has a different name or location.

In [ ]:
# Extrae el archivo ZIP desde tu Google Drive (elige el que hayas subido)
# Opción A: Si subiste los features pre-extraídos (Recomendado - 161 MB, tarda 5 segundos)
!unzip -q /content/drive/MyDrive/colab_training_data.zip -d ./

# Opción B: Si prefieres extraer el dataset completo de imágenes/videos crudos (Pesa ~8 GB)
# !unzip -q /content/drive/MyDrive/LSM_Dataset.zip -d ./data/


### 5. Train Models
Invoke `train_models.py` directly to train the static letters Random Forest model and the dynamic/words LSTM models.

In [ ]:
!python train_models.py

### 6. Export PyTorch LSTM Models to ONNX
To ensure compatibility and smooth loading in the desktop GUI application, run the cell below to export the trained PyTorch `.pth` models to ONNX format.

In [ ]:
import os
import json
import torch
import torch.onnx
from train_models import LSMLSTMModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def export_onnx(model_name, classes_name, output_onnx_name):
    model_path = os.path.join("models", model_name)
    classes_path = os.path.join("models", classes_name)
    output_path = os.path.join("models", output_onnx_name)
    
    if not os.path.exists(model_path):
        print(f"[Error] Model file {model_path} not found. Skipping export.")
        return
        
    with open(classes_path, 'r') as f:
        classes = json.load(f)
    num_classes = len(classes)
    
    # Initialize model
    model = LSMLSTMModel(input_dim=126, hidden_dim=64, num_layers=2, num_classes=num_classes)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    # Dummy input format: (batch_size=1, sequence_length=30, input_dimension=126)
    dummy_input = torch.randn(1, 30, 126).to(device)
    
    # Export to ONNX
    torch.onnx.export(
        model,
        dummy_input,
        output_path,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}},
        opset_version=11
    )
    print(f"[ONNX] Successfully exported {model_name} -> {output_onnx_name}")

# Export dynamic letters model
export_onnx("dynamic_letters_model.pth", "dynamic_letters_classes.json", "dynamic_letters_model.onnx")

# Export words model
export_onnx("words_model.pth", "words_classes.json", "words_model.onnx")

### 7. Copy Models Back to Google Drive
Save your trained models (`.joblib`, `.pth`, `.onnx` and mapping `.json` files) back to your Google Drive for backup and download.

In [ ]:
# Create a backup folder in your Drive
# !mkdir -p /content/drive/MyDrive/LSM_Trained_Models/

# Copy all files inside the models directory
# !cp -r models/* /content/drive/MyDrive/LSM_Trained_Models/
# print("All trained models, JSON classes, and ONNX configurations have been backed up to Drive!")